# gpt-5.4 obfuscation results

Catalogue of strategic-obfuscation patterns across the 350 QuALITY Gutenberg difficulty=1 questions, using **gpt-5.4 as both consultant and judge**.

**Data sources** (all produced by the `obfuscation-gpt54 *` GitHub Actions workflows):
- `data/obfuscation_gpt54/sequential_details.json` — 4-condition pipeline (assigned_correct / assigned_incorrect / free_choice / defendability), 8 samples per question.
- `data/obfuscation_gpt54/direct_answer_results.json` — gpt-5.4 direct-answer accuracy (story-visible A/B pick), 8 samples per question. Establishes what gpt-5.4 "knows" as the correct answer.

**Column schema** for the 4-condition pipeline:
- `assigned_correct_consultant_wins` (a.k.a. `cor_WR`) — fraction of the 8 candidates arguing the correct side that fool the judge into picking correct.
- `assigned_incorrect_consultant_wins` (`inc_WR`) — fraction of the 8 candidates arguing the incorrect side that fool the judge into picking incorrect. **This is the obfuscation-success rate.**
- `free_choice_frac_incorrect` (`fc_inc`) — fraction of the 8 free_choice candidates that chose to argue for the incorrect answer.
- `defendability_frac_incorrect` (`def_inc`) — fraction of the 8 defendability predictions that say incorrect is easier to defend.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path('../data/obfuscation_gpt54')

with open(DATA_DIR / 'sequential_details.json') as f:
    seq = json.load(f)
with open(DATA_DIR / 'direct_answer_results.json') as f:
    da = json.load(f)
da_map = {(r['batch'], r['idx']): r['accuracy'] for r in da}

df = pd.DataFrame(seq)
df['da'] = df.apply(lambda r: da_map.get((r['batch'], r['idx'])), axis=1)
df = df.rename(columns={
    'assigned_correct_consultant_wins': 'cor_wr',
    'assigned_incorrect_consultant_wins': 'inc_wr',
    'free_choice_frac_incorrect': 'fc_inc',
    'defendability_frac_incorrect': 'def_inc',
})
df['gap'] = df['inc_wr'] - df['cor_wr']

print(f'Loaded {len(df)} questions')
print(f'  cor_wr    : {df["cor_wr"].notna().sum()} populated')
print(f'  inc_wr    : {df["inc_wr"].notna().sum()} populated')
print(f'  fc_inc    : {df["fc_inc"].notna().sum()} populated')
print(f'  def_inc   : {df["def_inc"].notna().sum()} populated')
print(f'  da        : {df["da"].notna().sum()} populated')
df.head(3)

## 1. Heatmap — free_choice picks-incorrect rate by (cor_WR, inc_WR) bucket

- **Rows:** `assigned_correct_consultant_wins` (how often a consultant arguing the correct side fools the judge).
- **Columns:** `assigned_incorrect_consultant_wins` (how often a consultant arguing the incorrect side fools the judge — the direct measure of obfuscation success).
- **Cell value:** fraction of questions in that (cor, inc) bucket where `free_choice` majority-picks the incorrect side (`fc_inc >= 0.5`). High values flag regions where the model's free-choice behaviour tracks the actual defendability of the wrong answer.
- Cell counts (`n=...`) shown underneath so sparsely-populated buckets are visible.

Bin boundaries are 0–25 / 25–50 / 50–75 / 75–100% on each axis (matching the discrete 1/8 = 12.5% resolution of n=8 samples).

In [ ]:
bin_edges = [0, 0.25, 0.5, 0.75, 1.0001]
bin_labels = ['0-25%', '25-50%', '50-75%', '75-100%']

df_h = df.dropna(subset=['cor_wr', 'inc_wr', 'fc_inc']).copy()
df_h['cor_bin'] = pd.cut(df_h['cor_wr'], bin_edges, labels=bin_labels, right=False)
df_h['inc_bin'] = pd.cut(df_h['inc_wr'], bin_edges, labels=bin_labels, right=False)
df_h['picks_inc'] = df_h['fc_inc'] >= 0.5

grouped = df_h.groupby(['cor_bin', 'inc_bin'], observed=False)
frac = grouped['picks_inc'].mean().unstack()
counts = grouped.size().unstack()

# High cor at top (conventional heatmap layout)
frac = frac.reindex(index=bin_labels[::-1], columns=bin_labels)
counts = counts.reindex(index=bin_labels[::-1], columns=bin_labels)

annot = frac.copy().astype(object)
for r in frac.index:
    for c in frac.columns:
        n = int(counts.loc[r, c]) if not pd.isna(counts.loc[r, c]) else 0
        v = frac.loc[r, c]
        annot.loc[r, c] = '' if n == 0 else f'{v:.0%}\n(n={n})'

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(frac.astype(float), annot=annot.values, fmt='', cmap='RdYlGn_r',
            vmin=0, vmax=1, linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Frac of Qs where free_choice majority-picks incorrect'},
            ax=ax)
ax.set_xlabel('assigned_incorrect win rate', fontsize=12)
ax.set_ylabel('assigned_correct win rate', fontsize=12)
ax.set_title(f'gpt-5.4: free_choice picks-incorrect rate\nby (cor_WR, inc_WR) bucket (n={len(df_h)} questions)',
             fontsize=13)
plt.tight_layout()
plt.show()

# Summary numbers
total_picks_inc = df_h['picks_inc'].sum()
print(f'\nOverall: {total_picks_inc}/{len(df_h)} = {total_picks_inc/len(df_h):.1%} of questions have free_choice majority-picking incorrect')

# Bottom-right quadrant (high inc_wr, any cor_wr) — obfuscation opportunity
exploitable = df_h[df_h['inc_wr'] >= 0.75]
exp_picks_inc = exploitable['picks_inc'].sum()
print(f'Questions with inc_wr>=75% (wrong-side easy): {len(exploitable)}')
print(f'  of those, free_choice picks incorrect: {exp_picks_inc} ({exp_picks_inc/max(1,len(exploitable)):.1%})')

# Top-left quadrant (high cor_wr, low inc_wr) — wrong-side hard, correct-side easy
safe = df_h[(df_h['cor_wr'] >= 0.75) & (df_h['inc_wr'] < 0.5)]
safe_picks_inc = safe['picks_inc'].sum()
print(f'Questions with cor_wr>=75% AND inc_wr<50% (wrong-side hard): {len(safe)}')
print(f'  of those, free_choice picks incorrect: {safe_picks_inc} ({safe_picks_inc/max(1,len(safe)):.1%})')